# OneClickAI YOLO 모델 학습 및 실행
이 노트북은 Google Colab 환경에서 실행하도록 작성되었습니다.

**순서**
1. 패키지 설치
2. 데이터 업로드 및 압축 해제
3. 모델 학습 (학습/검증 8:2 자동 분리)
4. 이미지로 결과 확인
5. 웹캠 실시간 예측

## 1. 패키지 설치

In [ ]:
!pip install oneclickai -q

## 2. 데이터 업로드 및 압축 해제

이미지(`.jpg`, `.png`)와 라벨(`.txt`)이 한 폴더에 들어있는 zip 파일을 업로드해 주세요.

```
yolo_dataset.zip
└── yolo_dataset/
    ├── image1.jpg
    ├── image1.txt   ← 같은 이름의 이미지+라벨 쌍
    ├── image2.jpg
    ├── image2.txt
    └── ...
```

In [ ]:
from google.colab import files
import zipfile
import os

print("yolo_dataset.zip 파일을 업로드해 주세요.")
uploaded = files.upload()

for file_name in uploaded.keys():
    print(f"압축 해제 중: {file_name}")
    with zipfile.ZipFile(file_name, 'r') as zip_ref:
        zip_ref.extractall('.')

# 이미지+라벨 폴더 자동 탐색
def find_dataset(base='.'):
    for root, dirs, fs in os.walk(base):
        imgs = [f for f in fs if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        txts = [f for f in fs if f.endswith('.txt')]
        if imgs and txts:
            return root
    return None

dataset_path = find_dataset()
if dataset_path is None:
    raise FileNotFoundError("이미지/라벨 파일을 찾을 수 없습니다.")

print(f"데이터 폴더: {dataset_path}")

## 3. 모델 학습

학습/검증 데이터를 **자동으로 8:2 분리**해서 학습합니다.

- `epochs`: 전체 학습 반복 횟수
- `batch_size`: 한 번에 처리할 이미지 수 (GPU 메모리에 맞게 조절)
- `save_tflite`: `True`로 설정하면 `.tflite` 파일도 함께 저장됩니다

학습이 완료되면 날짜/시간 이름의 폴더에 모델 파일과 학습 그래프가 저장됩니다.

In [ ]:
from oneclickai.YOLO import fit_yolo_model

fit_yolo_model(
    dataset_path,
    dataset_path,
    epochs=30,
    batch_size=8,
    save_tflite=True
)

## 4. 이미지로 결과 확인

학습된 모델로 이미지를 예측하고 결과를 출력합니다.

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from oneclickai.YOLO import load_model, predict, draw_result

# 가장 최근 학습된 모델 자동 탐색
result_dirs = sorted([d for d in os.listdir('.') if os.path.isdir(d) and d[:4].isdigit()])
if not result_dirs:
    raise FileNotFoundError("학습된 모델을 찾을 수 없습니다. 먼저 3번 셀을 실행해 주세요.")

model_path = os.path.join(result_dirs[-1], 'yolo_model_best.keras')
print(f"모델 로드: {model_path}")
model = load_model(model_path)

# 이미지 5장 예측
sample_images = sorted([
    f for f in os.listdir(dataset_path)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])[:5]

for img_name in sample_images:
    img_path = os.path.join(dataset_path, img_name)
    image = cv2.imread(img_path)
    annotations = predict(model, image, conf=0.5, iou=0.5)
    result = draw_result(np.array(image), annotations)
    print(f"{img_name} — 감지된 객체: {len(annotations)}개")
    cv2_imshow(result)

## 5. 웹캠 실시간 예측

웹캠으로 프레임을 캡처해 YOLO 모델로 예측합니다.  
**실행하면 브라우저에서 카메라 권한을 요청합니다.**  
중단하려면 셀 왼쪽의 ■ 버튼을 누르거나 `Ctrl+M I`를 누르세요.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from base64 import b64decode
from IPython.display import clear_output
import numpy as np
import cv2

def capture_frame():
    data = eval_js('''
        new Promise(async (resolve) => {
            if (!window._yoloVideo || window._yoloVideo.readyState < 2) {
                window._yoloVideo = document.createElement('video');
                const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
                window._yoloVideo.srcObject = stream;
                await window._yoloVideo.play();
                await new Promise(r => setTimeout(r, 1500));
            }
            const canvas = document.createElement('canvas');
            canvas.width = window._yoloVideo.videoWidth;
            canvas.height = window._yoloVideo.videoHeight;
            canvas.getContext('2d').drawImage(window._yoloVideo, 0, 0);
            resolve(canvas.toDataURL('image/jpeg', 0.8));
        })
    ''')
    binary = b64decode(data.split(',')[1])
    arr = np.frombuffer(binary, dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


print("웹캠 실시간 예측 시작 (중단: 셀 왼쪽 ■ 버튼)")

frame_count = 0
while True:
    frame = capture_frame()
    if frame is None:
        break

    annotations = predict(model, frame, conf=0.5, iou=0.5)
    result = draw_result(np.array(frame), annotations)

    frame_count += 1
    clear_output(wait=True)
    print(f"프레임 #{frame_count} | 감지된 객체: {len(annotations)}개")
    cv2_imshow(result)